#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Modelos preentrenados, visualizando con las utilidades de ultralytics

In [23]:
from collections import defaultdict
import numpy as np
import csv                                                # NEU: zum Schreiben der CSV

model = YOLO('yolo11n.pt')
modelo_matriculas = YOLO('matriculas_yolo11n.pt')         # NEU (Kennzeichen): dein trainiertes Modell
CLASES_VEHICULO = [2, 3, 5, 7]                            # NEU (Kennzeichen): car, motorcycle, bus, truck

vid = cv2.VideoCapture("C0142.MP4")                       # GEÄNDERT: Video statt Webcam

# NEU: Writer für das Ergebnisvideo (gleiche Größe und Bildrate wie das Original)
fps = vid.get(cv2.CAP_PROP_FPS)
breite = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
hoehe = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("resultado.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (breite, hoehe))

ids_por_clase = defaultdict(set)                          # NEU: Tracking-IDs pro Klasse zum Zählen

# NEU: CSV öffnen und Kopfzeile schreiben (Spalten laut README des Professors)
archivo_csv = open("resultados.csv", "w", newline="", encoding="utf-8-sig")
csv_writer = csv.writer(archivo_csv, delimiter=";")
csv_writer.writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
                     "x1", "y1", "x2", "y2",
                     "matrícula_en_su_caso", "confianza_matricula",
                     "mx1", "my1", "mx2", "my2", "texto_matricula"])

fotograma = 0                                             # NEU: Bildzähler

while(True):
    ret, img = vid.read()
    if not ret:                                           # NEU: am Videoende stoppen
        break

    img_original = img.copy()                             # NEU (Kennzeichen): unbemalte Kopie für die Ausschnitte

    results = model.track(img, persist=True,
                          classes=[0,1,2,3,5,7],          # GEÄNDERT: + Motorrad, Bus, LKW
                          verbose=False)                  # NEU: keine Textausgabe pro Bild

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            if box.id is not None:
                track_id = str(int(box.id[0].tolist()))
            else:
                track_id = ''

            confidence = math.ceil((box.conf[0]*100))/100
            cls = int(box.cls[0])                         # GEÄNDERT: print-Zeilen entfernt

            if track_id != '':                            # NEU: ID in die Menge ihrer Klasse
                ids_por_clase[model.names[cls]].add(track_id)

            # NEU (Kennzeichen): Standardwerte, falls kein Kennzeichen gefunden wird
            hay_matricula = 0
            conf_mat, mx1, my1, mx2, my2 = "", "", "", "", ""

            # NEU (Kennzeichen): nur bei Fahrzeugen im Ausschnitt nach dem Kennzeichen suchen
            if cls in CLASES_VEHICULO:
                recorte = img_original[y1:y2, x1:x2]
                if recorte.shape[0] > 20 and recorte.shape[1] > 20:      # zu kleine Ausschnitte überspringen
                    res_mat = modelo_matriculas(recorte, conf=0.4, verbose=False)[0]
                    if len(res_mat.boxes) > 0:
                        mejor = int(res_mat.boxes.conf.argmax())        # Kennzeichen mit der höchsten Sicherheit
                        px1, py1, px2, py2 = res_mat.boxes.xyxy[mejor]
                        mx1, my1 = x1 + int(px1), y1 + int(py1)          # zurück auf das ganze Bild rechnen
                        mx2, my2 = x1 + int(px2), y1 + int(py2)
                        conf_mat = math.ceil((res_mat.boxes.conf[mejor]*100))/100
                        hay_matricula = 1
                        cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)   # gelber Rahmen

            # GEÄNDERT (Kennzeichen): Kennzeichen-Spalten werden jetzt gefüllt
            csv_writer.writerow([fotograma, model.names[cls], confidence, track_id,
                                 x1, y1, x2, y2,
                                 hay_matricula, conf_mat, mx1, my1, mx2, my2, ""])

            escala = int((cls / len(model.names)) * 255 * 3)   # GEÄNDERT: model.names
            if escala >= 255*2:
                R, G, B = 255, 255, escala - 255*2
            elif escala >= 255:
                R, G, B = 255, escala - 255, 0
            else:
                R, G, B = escala, 0, 0

            cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
            cv2.putText(img, track_id + ' ' + model.names[cls], [x1, y1],   # GEÄNDERT: model.names
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

    writer.write(img)                                     # NEU: Bild ins Ergebnisvideo schreiben

    anzeige = cv2.resize(img, None, fx=0.5, fy=0.5)       # NEU: nur Anzeige verkleinern
    cv2.imshow('Vid', anzeige)

    fotograma += 1                                        # NEU: nächstes Bild

    if cv2.waitKey(1) == 27:                              # GEÄNDERT: 1 statt 20 ms
        break

vid.release()
writer.release()                                          # NEU: Videodatei sauber abschließen
archivo_csv.close()                                       # NEU: CSV sauber abschließen
cv2.destroyAllWindows()

# NEU: Ergebnis der Zählung ausgeben
print("Número de objetos por clase:")
for clase, ids in ids_por_clase.items():
    print(clase, ":", len(ids))

Número de objetos por clase:
bus : 1
car : 6
truck : 1
person : 3


In [20]:
from collections import defaultdict
import numpy as np
import csv                                                # NEU: zum Schreiben der CSV

model = YOLO('yolo11n.pt')

vid = cv2.VideoCapture("C0142.MP4")                       # GEÄNDERT: Video statt Webcam

# NEU: Writer für das Ergebnisvideo (gleiche Größe und Bildrate wie das Original)
fps = vid.get(cv2.CAP_PROP_FPS)
breite = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
hoehe = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("resultado.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (breite, hoehe))

ids_por_clase = defaultdict(set)                          # NEU: Tracking-IDs pro Klasse zum Zählen

# NEU: CSV öffnen und Kopfzeile schreiben (Spalten laut README des Professors)
archivo_csv = open("resultados.csv", "w", newline="", encoding="utf-8-sig")
csv_writer = csv.writer(archivo_csv, delimiter=";")
csv_writer.writerow(["fotograma", "tipo_objeto", "confianza", "identificador_tracking",
                     "x1", "y1", "x2", "y2",
                     "matrícula_en_su_caso", "confianza_matricula",
                     "mx1", "my1", "mx2", "my2", "texto_matricula"])

fotograma = 0                                             # NEU: Bildzähler

while(True):
    ret, img = vid.read()
    if not ret:                                           # NEU: am Videoende stoppen
        break

    results = model.track(img, persist=True,
                          classes=[0,1,2,3,5,7],          # GEÄNDERT: + Motorrad, Bus, LKW
                          verbose=False)                  # NEU: keine Textausgabe pro Bild

    for r in results:
        boxes = r.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            if box.id is not None:
                track_id = str(int(box.id[0].tolist()))
            else:
                track_id = ''

            confidence = math.ceil((box.conf[0]*100))/100
            cls = int(box.cls[0])                         # GEÄNDERT: print-Zeilen entfernt

            if track_id != '':                            # NEU: ID in die Menge ihrer Klasse
                ids_por_clase[model.names[cls]].add(track_id)

            # NEU: eine Zeile pro Objekt; Kennzeichen-Spalten vorerst leer (kommt nach dem Training)
            csv_writer.writerow([fotograma, model.names[cls], confidence, track_id,
                                 x1, y1, x2, y2,
                                 0, "", "", "", "", "", ""])

            escala = int((cls / len(model.names)) * 255 * 3)   # GEÄNDERT: model.names
            if escala >= 255*2:
                R, G, B = 255, 255, escala - 255*2
            elif escala >= 255:
                R, G, B = 255, escala - 255, 0
            else:
                R, G, B = escala, 0, 0

            cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
            cv2.putText(img, track_id + ' ' + model.names[cls], [x1, y1],   # GEÄNDERT: model.names
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

    writer.write(img)                                     # NEU: Bild ins Ergebnisvideo schreiben

    anzeige = cv2.resize(img, None, fx=0.5, fy=0.5)       # NEU: nur Anzeige verkleinern
    cv2.imshow('Vid', anzeige)

    fotograma += 1                                        # NEU: nächstes Bild

    if cv2.waitKey(1) == 27:                              # GEÄNDERT: 1 statt 20 ms
        break

vid.release()
writer.release()                                          # NEU: Videodatei sauber abschließen
archivo_csv.close()                                       # NEU: CSV sauber abschließen
cv2.destroyAllWindows()

# NEU: Ergebnis der Zählung ausgeben
print("Número de objetos por clase:")
for clase, ids in ids_por_clase.items():
    print(clase, ":", len(ids))

Número de objetos por clase:
bus : 1
car : 6
truck : 1
person : 3
